# Agentic RAG over an Ontological Graph
### From Football PDFs to Natural Language Q&A — No OCR Required

This tutorial walks you through building an end-to-end **Agentic Retrieval-Augmented Generation (RAG)** system on top of a knowledge graph populated directly from PDF documents using vision AI.

| Step | What happens |
|---|---|
| **1 — Ontology** | Define the domain as Pydantic models |
| **2 — VLM Extraction** | GPT-4o reads PDF pages as images → structured JSON |
| **3 — Knowledge Graph** | JSON → Neo4J nodes & relationships (MERGE, idempotent) |
| **4 — Agentic RAG** | Natural language → Cypher → answer via tool-calling |

```
PDF ──► GPT-4o Vision ──► Structured JSON ──► Neo4J Graph ──► Agent ──► Answer
         (no OCR)           (Pydantic)        (MERGE)      (Cypher)
```

> **Domain:** Official match reports (*actes*) from the Federació Catalana de Futbol (FCF), written in Catalan.

---
## 0 · Environment Setup

**Local:** place a `.env` file at the repo root (copy `.env.example` and fill in your keys).  
**Google Colab:** add `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE` as Colab Secrets (*🔑 Runtime → Secrets*), then run the cell below.

In [ ]:
import sys, os, subprocess

# ── Detect environment ────────────────────────────────────────────────────────
try:
    from google.colab import userdata  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # System dependencies (poppler for pdf2image, graphviz for ontology diagram)
    subprocess.run(["apt-get", "install", "-y", "-q", "poppler-utils", "graphviz"],
                   capture_output=True, check=True)
    # Python dependencies
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   check=True)
    # ⚠️  If you haven't cloned the repo yet, run this first:
    # !git clone https://github.com/YOUR_ORG/summer-school.git
    # %cd summer-school

# ── Add repo root to Python path ─────────────────────────────────────────────
REPO_ROOT = os.path.abspath(".")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Repo root  : {REPO_ROOT}")

### Connectivity check

Verify that both services are reachable before proceeding.

In [ ]:
from src import setup

ok = setup.check()  # pings OpenAI and Neo4J, prints a Rich table
assert ok, "Fix the connection issues above before continuing."

---
## The Domain: FCF Match Reports

We work with **3 real match reports** from the FCF youth women's league (Jornada 29).  
Each PDF is a single page containing: team lineups, technical staff, goals, cards, stadium, and referee.  

The PDFs are in Catalan — the VLM handles the language automatically. Let's preview them.

In [ ]:
import io
from pdf2image import convert_from_path
from IPython.display import display, Image as IPImage
from src.common.paths import EXAMPLES_DIR

for n in [1, 2, 3]:
    pdf_path = EXAMPLES_DIR / f"example{n}.pdf"
    page = convert_from_path(str(pdf_path), dpi=90, first_page=1, last_page=1)[0]
    buf = io.BytesIO()
    page.save(buf, format="PNG")
    print(f"\n{'─'*60}")
    print(f"  example{n}.pdf")
    print(f"{'─'*60}")
    display(IPImage(buf.getvalue(), width=720))

---
## Step 1 · Ontology as Pydantic Models

The ontology serves double duty: it's both the **Python type system** for our code and the **JSON Schema** sent to GPT-4o as a structured output specification.  
Every field has a `description` that guides the VLM — no prompt engineering magic, just well-documented types.

In [ ]:
from src.ontology.schema import (
    MatchExtraction, Team, Player, LineupEntry, Coach,
    Stadium, Referee, Goal, Card,
    GoalType, CardColor, LineupRole, CardTargetKind,
)

# Every model uses extra='forbid' — the VLM cannot hallucinate extra fields
print("Top-level fields of MatchExtraction:")
for name, field in MatchExtraction.model_fields.items():
    annotation = str(field.annotation).replace("src.ontology.schema.", "")
    desc = field.description or ""
    print(f"  {name:20s}  {annotation:40s}  # {desc[:60]}")

In [ ]:
# A synthetic fixture — proves the schema validates correctly
sample = MatchExtraction(
    journey=29, competition="FCF", status="ACTA TANCADA",
    score_home=1, score_away=0,
    home=Team(
        name="Cirera",
        lineup=[LineupEntry(player=Player(name="Anna García", jersey=9), role=LineupRole.starter)],
        coaches=[Coach(name="Joan Puig", role_code="E")],
    ),
    away=Team(name="L'Estartit", lineup=[], coaches=[]),
    stadium=Stadium(name="Camp Municipal", address="Carrer Major, 1"),
    referee=Referee(name="Pere Mas", committee="Maresme"),
    goals=[Goal(minute=75, scoreline_home=1, scoreline_away=0,
                scorer_name="Anna García", scoring_team="home", type=GoalType.regular)],
    cards=[Card(minute=30, color=CardColor.yellow, target_kind=CardTargetKind.player,
                target_name="Laia Torres", team="away")],
)
assert len(sample.goals) == sample.score_home + sample.score_away
print("✅ Fixture validates — schema is correct.")

### Ontology diagram

The diagram below shows every node label and relationship type that will appear in the graph.  
It's generated programmatically from code — no manual drawing required.

In [ ]:
from src.ontology.visualize import render_ontology

diagram = render_ontology()
diagram  # renders inline — also saved to out/ontology.png

The same ontology as a **Mermaid diagram** (useful for documentation):

In [ ]:
from src.common.paths import OUT_DIR
from rich.syntax import Syntax
from rich.console import Console

mmd = (OUT_DIR / "ontology.mmd").read_text()
Console().print(Syntax(mmd, "markdown", theme="monokai"))

---
## Step 2 · VLM Extraction — No OCR

Each PDF page is converted to a PNG (220 DPI) and sent to `gpt-4o` as a base64-encoded image.  
The model must return a JSON object that **strictly validates** against the `MatchExtraction` schema — enforced via OpenAI's `response_format` structured outputs.

```
PDF → PNG → base64 → GPT-4o ──► JSON (validated by Pydantic)
                    ↑
          system prompt (domain description)
          + JSON Schema (MatchExtraction)
```

Results are **cached** in `data/extracted/` — re-running this cell never calls the API again.

In [ ]:
from src.extraction.pdf_to_images import convert_all
from IPython.display import display, Image as IPImage

image_paths = convert_all()  # cached in data/images/

# Preview one converted page
print("Sample converted page (example2.pdf):")
display(IPImage(str(image_paths[1]), width=680))

### The system prompt

The VLM receives a concise domain description in Spanish. Let's look at it:

In [ ]:
from src.extraction.vlm_extractor import _SYSTEM_PROMPT
from rich.panel import Panel
from rich.console import Console

Console().print(Panel(_SYSTEM_PROMPT, title="VLM system prompt", style="cyan"))

### Run extraction (3 PDFs)

In [ ]:
from src.extraction.runner import run

# force=False → skips already-extracted files (uses cache)
json_paths = run(force=False)

for p in json_paths:
    size = p.stat().st_size / 1024
    print(f"  ✓ {p.name}  ({size:.1f} KB)")

### Extracted data: match-by-match review

Let's inspect each extracted match — original image alongside the structured output.

In [ ]:
import json
from src.extraction.runner import inspect as vlm_inspect
from IPython.display import display

def show_match(n: int):
    img, data = vlm_inspect(n)
    display(img.resize((int(img.width * 0.45), int(img.height * 0.45))))

    print(f"\n{'═'*60}")
    print(f"  {data['home']['name']}  {data['score_home']} – {data['score_away']}  {data['away']['name']}")
    print(f"  Journey {data['journey']} · {data['competition']} · {data['status']}")
    print(f"  Stadium : {data['stadium']['name']}")
    print(f"  Referee : {data['referee']['name']} ({data['referee'].get('committee','')})")
    print(f"{'─'*60}")

    print(f"  Goals ({len(data['goals'])})")
    for g in data['goals']:
        icon = "⚽" if g['type'] == 'regular' else ("🔄" if g['type'] == 'own' else "⚡")
        print(f"    {icon} {g['minute']:>3}' {g['scorer_name']}  ({g['scoring_team']})  "
              f"{g['scoreline_home']}-{g['scoreline_away']}")

    print(f"  Cards ({len(data['cards'])})")
    for c in data['cards']:
        icon = "🟡" if c['color'] == 'yellow' else "🔴"
        print(f"    {icon} {c['minute']:>3}' {c['target_name']}  [{c['target_kind']}]  ({c['team']})")

    print(f"  Home lineup ({len(data['home']['lineup'])} players)")
    starters = [e for e in data['home']['lineup'] if e['role'] == 'starter']
    subs = [e for e in data['home']['lineup'] if e['role'] == 'sub']
    print(f"    Starters: {', '.join(e['player']['name'] for e in starters)}")
    print(f"    Subs    : {', '.join(e['player']['name'] for e in subs)}")

print("\n" + "▶" * 3 + "  MATCH 1  " + "◀" * 3)
show_match(1)

In [ ]:
print("\n" + "▶" * 3 + "  MATCH 2  " + "◀" * 3)
show_match(2)

In [ ]:
print("\n" + "▶" * 3 + "  MATCH 3  " + "◀" * 3 + "  (note: 2 coach cards!)")
show_match(3)

### Extraction quality check

Two automated validations:
1. **Goal count consistency** — `len(goals) == score_home + score_away`
2. **Scorer traceability** — every scorer's name appears in the respective team's lineup (after name normalization)

In [ ]:
from src.common.ids import normalize_name
from src.common.paths import EXTRACTED_DIR
from src.ontology.schema import MatchExtraction
from rich.table import Table
from rich.console import Console

table = Table(title="Extraction Quality")
table.add_column("File", style="bold")
table.add_column("Match")
table.add_column("Goals = score", justify="center")
table.add_column("Scorers in lineup", justify="center")
table.add_column("Coach cards", justify="center")

for n in [1, 2, 3]:
    m = MatchExtraction.model_validate_json(
        (EXTRACTED_DIR / f"example{n}.json").read_text()
    )
    all_players = {normalize_name(e.player.name) for e in m.home.lineup + m.away.lineup}
    goals_ok = len(m.goals) == m.score_home + m.score_away
    scorers_ok = all(normalize_name(g.scorer_name) in all_players for g in m.goals)
    coach_cards = sum(1 for c in m.cards if c.target_kind.value == "coach")

    table.add_row(
        f"example{n}.json",
        f"{m.home.name} {m.score_home}-{m.score_away} {m.away.name}",
        "✅" if goals_ok else "❌",
        "✅" if scorers_ok else "❌",
        str(coach_cards) if coach_cards else "—",
    )

Console().print(table)

---
## Step 3 · Building the Knowledge Graph

We ingest the 3 JSON files into **Neo4J AuraDB** using exclusively `MERGE` statements — making every write **idempotent**: running ingestion twice leaves exactly the same graph.

**Uniqueness constraints** are applied first so that duplicate nodes are impossible at the database level.

In [ ]:
from src.graph.constraints import apply as apply_constraints

apply_constraints()  # CREATE CONSTRAINT ... IF NOT EXISTS (safe to re-run)

### Deterministic node IDs

Node identity is resolved by **normalizing names** to a canonical form: strip accents, uppercase, collapse whitespace.  
The same function is used in both the extractor and the ingestor — same input always yields the same ID.

In [ ]:
from src.common.ids import normalize_name, player_id, team_id, match_id

examples = [
    ("PUIGGROS CAMA, CARLA",  player_id),
    ("L'Estartit, U.E. A",    team_id),
    ("Cirera",                 team_id),
    ("García Díez, Ismael",    normalize_name),
]
print(f"{'Input':<35}  {'Normalized ID'}")
print("─" * 70)
for raw, fn in examples:
    print(f"{raw:<35}  {fn(raw)}")

In [ ]:
from src.graph.ingest import ingest_all

counts = ingest_all()  # reads data/extracted/*.json, MERGEs everything

In [ ]:
from rich.table import Table
from rich.console import Console

table = Table(title="Node counts after ingestion")
table.add_column("Label", style="bold cyan")
table.add_column("Count", justify="right")
for label, count in sorted(counts.items()):
    table.add_row(label, str(count))
Console().print(table)

# Quick sanity checks
assert counts["Match"] == 3,  "Expected 3 matches"
assert counts["Goal"]  == 15, "Expected 15 goals (6 + 1 + 8)"
assert counts["Card"]  == 7,  "Expected 7 cards (1 + 1 + 5)"
print("\n✅ All counts correct.")

### Idempotency test

Running ingestion a second time must leave the **exact same counts**.

In [ ]:
counts2 = ingest_all()
assert counts2 == counts, f"Idempotency failed!\nRun 1: {counts}\nRun 2: {counts2}"
print("✅ Idempotency confirmed — same counts after second ingestion.")

### Interactive graph visualization

The visualization is built with **pyvis** — a force-directed graph rendered as self-contained HTML.  
Node colors reflect labels; hover over nodes and edges for details.

> You can also explore the graph interactively at [console.neo4j.io](https://console.neo4j.io).

In [ ]:
from src.graph.visualize import render_graph

render_graph(limit=300)  # returns IFrame — renders inline in Jupyter

---
## Step 4 · Agentic RAG with Tool-Calling

The agent is a **GPT-4o** loop that translates natural language questions into Cypher queries using three tools:

| Tool | What it does |
|---|---|
| `get_graph_schema` | Returns the full schema (labels, relationships, properties, Cypher tips) |
| `validate_cypher` | Runs `EXPLAIN <query>` — checks syntax without executing |
| `run_cypher` | Executes a read-only Cypher query against Neo4J |

**Read-only guard:** any query containing `CREATE / MERGE / DELETE / SET / REMOVE / DROP / CALL` is rejected before reaching Neo4J.

**Recovery loop:** if a query returns empty results, the agent relaxes its filters and retries (up to `max_iterations`).

In [ ]:
from src.graph.schema_intro import graph_schema_summary
from rich.panel import Panel
from rich.console import Console

schema_text = graph_schema_summary()
Console().print(Panel(schema_text, title="What the agent sees as graph schema", style="green"))

### Read-only guard in action

Before running demo questions, let's verify that write operations are rejected:

In [ ]:
from src.agent.tools import run_cypher

blocked = run_cypher("CREATE (n:Test {id: 'hacked'}) RETURN n")
print("Attempted CREATE:", blocked)
assert blocked["error"] is not None and "CREATE" in blocked["error"]
print("\n✅ Write operations correctly blocked.")

### Agent architecture

Each call to `ask()` runs this loop:

```
messages = [system_prompt, user_question]
for iteration in range(max_iterations):
    response = gpt-4o(messages, tools=TOOL_SPECS)
    if response has tool_calls:
        execute each tool → append results to messages
    else:
        return response.content  ← final answer
```

The full conversation trace is saved to `out/agent_traces/{timestamp}.json`.

---
## Demo Questions

Six questions designed to test different capabilities of the system.

### Q1 · Direct lookup

Simple match score retrieval. Watch how the agent handles the apostrophe in *L'Estartit* — it will fail on the first attempt and self-correct.

In [ ]:
from src.agent.agent import ask
from rich.panel import Panel
from rich.console import Console

result = ask("¿Cuál fue el marcador entre Cirera y L'Estartit?")

Console().print(Panel(result.answer, title="Answer", style="bold green"))
print(f"\nCypher attempts: {len(result.cypher_attempts)}")
for i, attempt in enumerate(result.cypher_attempts, 1):
    status = "✅" if attempt["ok"] else "❌"
    print(f"  [{i}] {status} rows={attempt['rows']}  error={attempt['error']}")

### Q2 · Aggregation

The agent needs to traverse `Match → Goal → Player` and aggregate with `COUNT`.

In [ ]:
result = ask("¿Qué jugadora marcó más goles en la jornada 29?")
Console().print(Panel(result.answer, title="Answer", style="bold green"))

### Q3 · Multi-hop traversal

The agent must first find which team conceded 6 goals, then find the stadium where that match was played.

In [ ]:
result = ask("¿En qué estadio jugó el equipo que encajó 6 goles?")
Console().print(Panel(result.answer, title="Answer", style="bold green"))

### Q4 · Temporal filter

Filter on a node property (`minute < 30`) across all matches.

In [ ]:
result = ask("Lista los goles marcados antes del minuto 30.")
Console().print(Panel(result.answer, title="Answer", style="bold green"))

### Q5 · Honesty under incomplete data

The graph models **goals** (including penalty goals as `type='penalty'`), but does **not** model missed/saved penalties.  
A well-designed agent should recognise the limits of the data and say so.

In [ ]:
result = ask("¿Cuántos penaltis se fallaron?")
Console().print(Panel(result.answer, title="Answer", style="bold green"))

### Q6 · Mixed edge traversal

This tests the `GIVEN_TO_COACH` relationship — a less common path in the graph.  
Example 3 (Inter Granollers vs Aro) has 2 coach cards, which the VLM correctly captured.

In [ ]:
result = ask("¿Algún entrenador o delegado recibió tarjeta?")
Console().print(Panel(result.answer, title="Answer", style="bold green"))

### Inspecting a full agent trace

Every conversation is persisted as JSON in `out/agent_traces/`. Let's inspect the latest one:

In [ ]:
import json
from src.common.paths import TRACES_DIR
from rich.syntax import Syntax
from rich.console import Console

latest_trace = sorted(TRACES_DIR.glob("*.json"))[-1]
trace = json.loads(latest_trace.read_text())

print(f"Trace file : {latest_trace.name}")
print(f"Question   : {trace['question']}")
print(f"Iterations : {len([m for m in trace['messages'] if m.get('role') == 'tool'])}")
print(f"Answer     : {trace['answer']}")
print(f"\nCypher attempts:")
for i, a in enumerate(trace['cypher_attempts'], 1):
    status = "✅" if a['ok'] else "❌"
    print(f"  [{i}] {status} rows={a['rows']}")
    Console().print(Syntax(a['query'], "cypher", theme="monokai", word_wrap=True))

---
## Interactive REPL

Ask your own questions. Type `exit` to stop.

In [ ]:
from src.agent.agent import ask
from rich.panel import Panel
from rich.console import Console
import sys

console = Console()

# Guard: input() requires interactive stdin (not available in nbconvert / CI)
_interactive = sys.stdin is not None and hasattr(sys.stdin, "isatty") and sys.stdin.isatty()

if _interactive:
    console.print("[bold]Type your question (or 'exit' to stop):[/bold]")
    while True:
        try:
            question = input("\nQuestion: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not question or question.lower() in {"exit", "quit", "q"}:
            break
        result = ask(question)
        console.print(Panel(result.answer, title="Answer", style="bold green"))
else:
    console.print("[yellow]REPL skipped in non-interactive mode.[/yellow]")
    console.print("[dim]Run manually: python tutorial.py repl[/dim]")

---
## Conclusion & Key Takeaways

| Concept | How it's applied here |
|---|---|
| **Vision LLM as parser** | GPT-4o reads PDFs as images — no OCR, no layout heuristics |
| **Structured outputs** | Pydantic schema → JSON Schema → enforced by OpenAI API |
| **Knowledge graph** | Neo4J with MERGE — idempotent, queryable, explainable |
| **Agentic RAG** | Tool-calling loop: schema → validate → run → recover → answer |
| **Read-only safety** | Regex guard prevents any write query from the agent |
| **Colab-compatible** | No Docker, no local Neo4J — AuraDB Free + pip installs |

### What to explore next

- **Scale up**: load a full season of match reports and query across time
- **Richer ontology**: add substitutions, match events, weather, attendance
- **Vector + graph hybrid**: embed player names for fuzzy matching on top of exact Cypher
- **Multi-document reasoning**: questions that span several matches or seasons
- **Evaluation**: build a benchmark of Q&A pairs and measure answer accuracy automatically